In [ ]:
import torch
from dataset_utils import *
import numpy as np
import umap
from pipeline_utils.pipeline_utils import create_profile_vector, create_tweet_vectors
from sklearn.preprocessing import RobustScaler
from transformers import DistilBertTokenizer, AutoModel
import joblib

In [ ]:
def create_raw_embedding(data_loader, device = "cuda"):
    # -- initial setup
    tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
    model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
    scaler = RobustScaler(with_centering=False)

    # -- training loop
    ground_truth_labels = []
    profile_vectors = []
    tweet_vectors = []
    for i, sample in enumerate(data_loader):
        # get feature vectors
        profile_embed = create_profile_vector(sample.user_data)
        tweet_embeds = create_tweet_vectors(sample.tweet_data, tokenizer, model, max_tweets= 300, batch_size = 300)
        # pool tweet vectors
        tweet_vec = torch.mean(tweet_embeds, dim=0, dtype=torch.float32)

        # append feature vectors and ground truth
        profile_vectors.append(profile_embed)
        tweet_vectors.append(tweet_vec)
        ground_truth_labels.append(sample.label)
        print(f"\rIteration: {i} | {sample.label}", end="")
    # scale profile vectors
    scaled_profile_vectors = scaler.fit_transform(profile_vectors)


    return tweet_vectors, scaled_profile_vectors, ground_truth_labels

def save_processed_dataset(train_dataset, test_dataset, dataset_name, umap_reducer, train_umap = False, root_path = "../datasets/ProcessedDatasets/", device="cuda"):
    # process training dataset
    tweet_vectors, profile_vectors, labels = create_raw_embedding(train_dataset, device = "cuda")
    if train_umap: train_reduced_tweet_vectors = umap_reducer.fit_transform(X=tweet_vectors, y=torch.tensor(labels))
    else: train_reduced_tweet_vectors = umap_reducer.transform(tweet_vectors)
    train_reduced_tweet_vectors = np.nan_to_num(train_reduced_tweet_vectors, nan=0.0, posinf=0.0, neginf=0.0)
    train_embedding_features = [np.concatenate((t1, t2), axis=0) for t1, t2 in zip(train_reduced_tweet_vectors, profile_vectors)]

    # safe values
    embeddings = torch.tensor(np.array(train_embedding_features))
    ground_truths = torch.tensor(np.array(labels))
    embeddings = embeddings.cpu().detach()
    ground_truths = ground_truths.cpu().detach()

    torch.save(embeddings, f"{root_path}{dataset_name}TrainEmbed.pt")
    torch.save(ground_truths, f"{root_path}{dataset_name}TrainLabel.pt")

    # process test dataset
    tweet_vectors, profile_vectors, labels = create_raw_embedding(test_dataset, device = "cuda")
    reduced_tweet_vectors = umap_reducer.transform(tweet_vectors)
    reduced_tweet_vectors = np.nan_to_num(reduced_tweet_vectors, nan=0.0, posinf=0.0, neginf=0.0)
    embedding_features = [np.concatenate((t1, t2), axis=0) for t1, t2 in zip(reduced_tweet_vectors, profile_vectors)]

    # safe values
    embeddings = torch.tensor(np.array(embedding_features))
    ground_truths = torch.tensor(np.array(labels))
    embeddings = embeddings.cpu().detach()
    ground_truths = ground_truths.cpu().detach()

    torch.save(embeddings, f"{root_path}{dataset_name}TestEmbed.pt")
    torch.save(ground_truths, f"{root_path}{dataset_name}TestLabel.pt")

In [ ]:
# Dataset: Cresci17
dim_reducer = umap.UMAP(n_components=38, n_neighbors=20, min_dist=0.1, metric="cosine", target_metric="categorical", target_weight=0.25)

train_dataset = InterleavedIterableDataset([
            Cresci17(Cresci17SetTypes.GENUINE_USER, "train", root="../datasets", custom_label=0),
            Cresci17(Cresci17SetTypes.FAKE_FOLLOWER, "train", root="../datasets", custom_label=1),
            Cresci17(Cresci17SetTypes.SOCIAL_SPAM_1, "train", root="../datasets", custom_label=2),
            Cresci17(Cresci17SetTypes.SOCIAL_SPAM_2, "train", root="../datasets", custom_label=2),
            Cresci17(Cresci17SetTypes.SOCIAL_SPAM_3, "train", root="../datasets", custom_label=2),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_1, "train", root="../datasets", custom_label=3),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_2, "train", root="../datasets", custom_label=3),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_3, "train", root="../datasets", custom_label=3),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_4, "train", root="../datasets", custom_label=3),
        ], "Random")

test_dataset = InterleavedIterableDataset([
            Cresci17(Cresci17SetTypes.GENUINE_USER, "test", root="../datasets", custom_label=0),
            Cresci17(Cresci17SetTypes.FAKE_FOLLOWER, "test", root="../datasets", custom_label=1),
            Cresci17(Cresci17SetTypes.SOCIAL_SPAM_1, "test", root="../datasets", custom_label=2),
            Cresci17(Cresci17SetTypes.SOCIAL_SPAM_2, "test", root="../datasets", custom_label=2),
            Cresci17(Cresci17SetTypes.SOCIAL_SPAM_3, "test", root="../datasets", custom_label=2),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_1, "test", root="../datasets", custom_label=3),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_2, "test", root="../datasets", custom_label=3),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_3, "test", root="../datasets", custom_label=3),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_4, "test", root="../datasets", custom_label=3),
        ], "Random")

save_processed_dataset(train_dataset, test_dataset, "Cresci17", dim_reducer, True, "../datasets/ProcessedDatasets/", device="cuda")
joblib.dump(dim_reducer, "../datasets/ProcessedDatasets/UmapModel.joblib")

In [ ]:
# Dataset: Cresci18
dim_reducer = joblib.load("../datasets/ProcessedDatasets/UmapModel.joblib")

train_dataset = Cresci18("train", root="./datasets", use_only_labelled=True, label_mapping=[5,0,"unlabelled"])
test_dataset = Cresci18("test", root="./datasets", use_only_labelled=True, label_mapping=[5,0,"unlabelled"])

save_processed_dataset(train_dataset, test_dataset, "Cresci18", dim_reducer, False, "../datasets/ProcessedDatasets/", device="cuda")

In [ ]:
# Dataset: Caverlee11
dim_reducer = joblib.load("../datasets/ProcessedDatasets/UmapModel.joblib")

train_dataset = Caverlee11("train", root="../datasets", label_mapping=[4,0])
test_dataset = Caverlee11("test", root="../datasets", label_mapping=[4,0])

save_processed_dataset(train_dataset, test_dataset, "Caverlee11", dim_reducer, False, "../datasets/ProcessedDatasets/", device="cuda")

In [ ]:
# Dataset: Twibot20
dim_reducer = joblib.load("../datasets/ProcessedDatasets/UmapModel.joblib")

train_dataset = Twibot20("train", root="../datasets", label_mapping=[6, 0])
test_dataset = Twibot20("test", root="../datasets", label_mapping=[6, 0])

save_processed_dataset(train_dataset, test_dataset, "Twibot20", dim_reducer, False, "../datasets/ProcessedDatasets/", device="cuda")

In [ ]:
# Dataset: Twibot22
dim_reducer = joblib.load("../datasets/ProcessedDatasets/UmapModel.joblib")

train_dataset = Twibot22Improved("train", root="../datasets", label_mapping=[7,0], sub_sample_size=40000)
test_dataset = Twibot22Improved("test", root="../datasets", label_mapping=[7,0])

save_processed_dataset(train_dataset, test_dataset, "Twibot22", dim_reducer, False, "../datasets/ProcessedDatasets/", device="cuda")